# Box-Score SPM → Prior → Final RAPM

## Architecture

```
Box features (3-yr rolling avg)
        │
        ▼
  Box model (Ridge / XGBoost)  — separate O-RAPM & D-RAPM models
  Target: 3-year RAPM
        │
        ├──► Box-only score  (Metric A)
        │
        ├──► write_prior_csv()  → JE_rapm/prior.csv
        │         (for rapm_with_prior.py when MySQL DB is available)
        │
        └──► Approximate prior blend  (Metric B, tested at α=2000/4000/6000)
               Final = (Box × α + Raw_1yr × N) / (α + N)
```

## Team Context Features
Team ORtg and DRtg are scraped directly from basketball-reference  
(`scrape_team_ratings` → `leagues/NBA_{year}.html`).  
**Computing them from player box data is wrong** because summing player  
OffPoss double-counts shared possessions among teammates.

## Retrodiction Test (Krishna Narsu / LEBRON methodology)

For each year Y → Y+1:
1. Predict player score from year-Y features
2. Multiply by **actual year-Y+1 minutes**
3. Sum by team → compare to **actual year-Y+1 wins** (scraped from basketball-reference)
4. Compute **Pearson r²** (scale-invariant, matches EPM/LEBRON reporting)

Threshold assignment:
- `< 200 min` in Y+1 → **replacement value** (median RAPM of low-minute players)
- No year-Y score (rookies) → **replacement + 0.5** (near-replacement)

## Correct RAPM-with-Prior Methodology (JE approach)

The *proper* way to use box predictions as a prior in RAPM is:
```
offset   = X @ μ           (prior's contribution to each possession)
y_adj    = y − offset       (residual the ridge explains)
raw_coef = Ridge(X, y_adj, α)
final    = raw_coef + μ     (total = residual + prior)
```
This is ridge regularisation toward μ instead of toward 0.  
It requires the matchup X matrix (lives in MySQL DB).  
`write_prior_csv()` exports μ in the expected format for `rapm_with_prior.py`.

## Approximate Blend (used here, without MySQL)
```
N     = player_minutes × 2.05   (≈ single-season possessions)
Final = (Box_Prior × α + Raw_1yr × N) / (α + N)
```
This approximates the ridge-with-prior under the assumption X'X ≈ N·I.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')

HERE = pathlib.Path().resolve()
sys.path.insert(0, str(HERE / 'src'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score as sk_r2

from load_data  import load_box_scores, load_synergy_playtypes
from zts        import compute_zts
from features   import compute_all_features
from load_rapm  import load_rapm_3year, load_rapm_1year, load_rapm_with_prior
from model import (
    scrape_team_ratings, scrape_team_wins,
    compute_contextual_features,
    assemble_model_data, compute_rolling_features,
    compute_replacement_value, build_train_matrix,
    train_ridge, train_xgboost, retrodiction_test,
    compute_prior_blend, write_prior_csv,
    ridge_importance, xgb_importance,
    MODEL_FEATURES, OFFENSE_FEATURES, DEFENSE_FEATURES,
    REPL_MIN_THRESH, ROOKIE_BONUS,
)

SITE_DATA = HERE.parent / 'site_Data'
JE        = HERE.parent / 'JE_rapm'

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.3f}'.format)
print('Imports OK')

Imports OK


## 1. Load All Data

In [2]:
print('Box scores + zTS...')
box     = load_box_scores(min_year=2014, max_year=2025)
synergy = load_synergy_playtypes()
zts_df  = compute_zts(box, synergy, min_minutes=0, min_syn_poss=0)

print('Engineered features...')
feats = compute_all_features(box)

print('RAPM data...')
rapm3   = load_rapm_3year(min_season=2016)
rapm1   = load_rapm_1year(min_season=2014)
rapm_pr = load_rapm_with_prior()   # pre-computed prior+blend for benchmark

print(f'  3yr RAPM: {rapm3.shape}, seasons {rapm3.Season.min()}–{rapm3.Season.max()}')
print(f'  1yr RAPM: {rapm1.shape}, Prior RAPM: {rapm_pr.shape}')

print('Team ratings (scraped from basketball-reference)...')
ratings = scrape_team_ratings(seasons=list(range(2015, 2025)))
tw      = ratings[['Team','Season','team_wins']].dropna(subset=['team_wins']).copy()
print(f'  {len(ratings)} team-seasons  |  ORtg range: {ratings.team_ortg.min():.1f}–{ratings.team_ortg.max():.1f}')
print('\n2024 team ORtg / DRtg (sanity check — should be ~110–125):')
display(ratings[ratings.Season==2024].nlargest(10,'team_ortg')[['Team','team_ortg','team_drtg','team_wins']])

Box scores + zTS...


Using cached file: /Users/eadebayo/Documents/Projects/Sports Analytics/New SPM/zts/data/raw/playtype_raw.csv
Engineered features...


RAPM data...
  3yr RAPM: (6692, 6), seasons 2016–2024
  1yr RAPM: (5796, 6), Prior RAPM: (4322, 10)
Team ratings (scraped from basketball-reference)...
  300 team-seasons  |  ORtg range: 95.5–123.2

2024 team ORtg / DRtg (sanity check — should be ~110–125):


,Team,team_ortg,team_drtg,team_wins
270,BOS,123.200,111.600,64
279,IND,121.000,118.000,47
271,OKC,119.500,112.100,57
276,LAC,118.800,115.400,51
273,DEN,118.500,113.000,57
281,MIL,118.400,115.800,49
274,NYK,118.200,113.400,50
280,GSW,117.800,115.200,46
278,PHO,117.600,114.600,49
283,DAL,117.600,115.400,50


## 2. Feature Assembly

In [3]:
# Get Team column from LEBRON data (needed for contextual features)
leb_raw = pd.read_csv(str(SITE_DATA / 'lebron.csv'))
leb_raw = leb_raw.rename(columns={'NBA ID':'PLAYER_ID','year':'Season','team':'Team'})
if leb_raw.columns.duplicated().any():
    leb_raw = leb_raw.loc[:, ~leb_raw.columns.duplicated(keep='last')]
for col in ['PLAYER_ID','Season']:
    leb_raw[col] = pd.to_numeric(leb_raw[col], errors='coerce')
leb_raw = leb_raw.dropna(subset=['PLAYER_ID','Season'])
leb_raw['PLAYER_ID'] = leb_raw['PLAYER_ID'].astype(int)
leb_raw['Season']    = leb_raw['Season'].astype(int)
leb_raw['Team']      = leb_raw['Team'].str.upper().str.strip()
leb_slim = leb_raw[['PLAYER_ID','Season','Team']].drop_duplicates(['PLAYER_ID','Season'])

box_t = box.merge(leb_slim, on=['PLAYER_ID','Season'], how='left')

# Pass scraped team ratings (ORtg / DRtg) — NOT team wins — for contextual features.
# Using player box data to compute team ORtg is wrong because summing player
# OffPoss overcounts shared possessions (5 teammates each "have" 100 possessions
# → summed as 500, not 100).
ctx = compute_contextual_features(box_t, ratings)

# Sanity-check: team_ortg_ctx should reflect real per-team ORtg (~10–25 range
# after weighting by player minute share ≈ 10–15% of 82×240=19680 team min)
print('Sample team_ortg_ctx values (should be 10–20 range, reflecting ORtg×min_share):')
print(ctx[ctx.Season==2024].dropna(subset=['team_ortg_ctx'])
      .nlargest(6,'team_ortg_ctx')[['PLAYER_ID','team_ortg','team_ortg_ctx']].to_string(index=False))
print()

raw   = assemble_model_data(box_t, zts_df, feats, ctx, rapm3, rapm1)
avail = [c for c in MODEL_FEATURES if c in raw.columns]

print(f'Wide table: {raw.shape}')
print(f'Feature cols: {avail}')
print(f'3yr RAPM coverage: {raw.RAPM_3y.notna().sum()} / {len(raw)}')
print(f'1yr RAPM coverage: {raw.RAPM_1y.notna().sum()} / {len(raw)}')

Sample team_ortg_ctx values (should be 10–20 range, reflecting ORtg×min_share):
 PLAYER_ID  team_ortg  team_ortg_ctx
   1628973    118.200         20.243
   1627783    112.300         20.111
   1628404    118.200         20.102
   1630178    116.900         19.683
    203471    112.300         18.810
    202699    116.900         17.749

Wide table: (6293, 54)
Feature cols: ['PTS_per100', 'zTS', 'ThreePtP', 'Creation', 'Load', 'cTOV_pct', 'PasserRating', 'TightThree_per100', 'RimAssists_per100', 'Assisted2s_pct', 'Assisted3s_pct', 'PotentialAst_per100', 'SecondaryAst_per100', 'RimPointsSaved', 'C6_Diff_pct', 'fTOV_per100', 'Deflections_per100', 'DREB_Uncontest_per100', 'OREB_Contest_per100', 'GP_pct', 'team_ortg_ctx', 'team_drtg_ctx']
3yr RAPM coverage: 4804 / 6293
1yr RAPM coverage: 5778 / 6293


In [4]:
# 3-year rolling average
rolled = compute_rolling_features(raw, window=3, feature_cols=avail)
print(f'Rolled table: {rolled.shape}')

repl_val   = compute_replacement_value(rapm3, box)
rookie_val = repl_val + ROOKIE_BONUS
print(f'Replacement RAPM:  {repl_val:.3f}')
print(f'Rookie RAPM:       {rookie_val:.3f}')

Rolled table: (6293, 54)
Replacement RAPM:  -0.283
Rookie RAPM:       0.217


## 3. Model Training

**Target**: 3-year RAPM (most stable player value signal)  
**Inputs**: 3-year rolling average of box + tracking + zTS features  
**Filter**: ≥ 200 minutes in season

In [5]:
X_df, X, y, gmed = build_train_matrix(rolled, avail, target='RAPM_3y', min_minutes=200)
print(f'Training set: {X.shape[0]:,} player-seasons, {X.shape[1]} features')
print(f'3yr RAPM range: {y.min():.2f} to {y.max():.2f}, median: {np.median(y):.2f}')

Training set: 3,823 player-seasons, 22 features
3yr RAPM range: -7.32 to 9.59, median: 0.18


In [6]:
print('=== Total RAPM models ===')
print('Training Ridge (total RAPM)...')
lin_m, lin_sc = train_ridge(X, y, alpha=1.0)
print('Training XGBoost (total RAPM)...')
xgb_m = train_xgboost(X, y)

Xs = lin_sc.transform(X)
print(f'Ridge in-sample  R²: {sk_r2(y, lin_m.predict(Xs)):.4f}')
print(f'XGBoost in-sample R²: {sk_r2(y, xgb_m.predict(X)):.4f}')

# --- Separate O / D models (for write_prior_csv O/D split) ---
print('\n=== Offensive RAPM model ===')
o_avail = [c for c in OFFENSE_FEATURES if c in rolled.columns]
X_o_df, X_o, y_o, gmed_o = build_train_matrix(rolled, o_avail, target='ORAPM_3y', min_minutes=200)
print(f'Training set: {X_o.shape[0]:,} player-seasons, {X_o.shape[1]} features')
o_lin_m, o_lin_sc = train_ridge(X_o, y_o, alpha=1.0)
o_xgb_m           = train_xgboost(X_o, y_o)
print(f'Ridge O-RAPM in-sample R²: {sk_r2(y_o, o_lin_m.predict(o_lin_sc.transform(X_o))):.4f}')
print(f'XGB   O-RAPM in-sample R²: {sk_r2(y_o, o_xgb_m.predict(X_o)):.4f}')

print('\n=== Defensive RAPM model ===')
d_avail = [c for c in DEFENSE_FEATURES if c in rolled.columns]
X_d_df, X_d, y_d, gmed_d = build_train_matrix(rolled, d_avail, target='DRAPM_3y', min_minutes=200)
print(f'Training set: {X_d.shape[0]:,} player-seasons, {X_d.shape[1]} features')
d_lin_m, d_lin_sc = train_ridge(X_d, y_d, alpha=1.0)
d_xgb_m           = train_xgboost(X_d, y_d)
print(f'Ridge D-RAPM in-sample R²: {sk_r2(y_d, d_lin_m.predict(d_lin_sc.transform(X_d))):.4f}')
print(f'XGB   D-RAPM in-sample R²: {sk_r2(y_d, d_xgb_m.predict(X_d)):.4f}')

=== Total RAPM models ===
Training Ridge (total RAPM)...
Training XGBoost (total RAPM)...


Ridge in-sample  R²: 0.4782
XGBoost in-sample R²: 0.7927

=== Offensive RAPM model ===
Training set: 3,823 player-seasons, 15 features


Ridge O-RAPM in-sample R²: 0.4999
XGB   O-RAPM in-sample R²: 0.7911

=== Defensive RAPM model ===
Training set: 3,823 player-seasons, 8 features


Ridge D-RAPM in-sample R²: 0.2102
XGB   D-RAPM in-sample R²: 0.5801


## 4. Feature Importance + Ridge vs XGB Diagnosis

**Why Ridge underperforms XGB in retrodiction:**
- Ridge is a linear model — it can't capture non-linear interactions (e.g. "high Creation AND high zTS together = extra boost")
- With many correlated features, Ridge distributes coefficients broadly rather than picking the right signal
- `team_ortg_ctx` tends to dominate Ridge because it has a wide numeric range (0–20), making its coefficient misleadingly large even after standardization
- XGB learns feature thresholds and interactions, making it more robust to multicollinearity

**Why values differ between Ridge and XGB for the same player:**
- Ridge rates players linearly by summing weighted features → favors players with consistently high values across all offensive metrics (e.g. high-usage scorers/creators)
- XGB picks up on non-linear combinations → can rate defensive specialists, efficient low-usage players, or two-way players differently

In [7]:
ridge_imp = ridge_importance(lin_m, lin_sc, avail)
xgb_imp   = xgb_importance(xgb_m, avail)

print('Ridge — standardised coefficients (features scaled to unit variance):')
print('  Large team_ortg/drtg_ctx coefficients → Ridge is partly predicting team quality')
display(ridge_imp.head(15))

print('\nXGBoost — feature gain (captures non-linear importance, more player-specific):')
display(xgb_imp.head(15))

# Side-by-side rank comparison
ridge_rank = ridge_imp.reset_index(drop=True).reset_index().rename(columns={'index':'Ridge_rank','Feature':'Feature'})
xgb_rank   = xgb_imp.reset_index(drop=True).reset_index().rename(columns={'index':'XGB_rank'})
ridge_rank['Ridge_rank'] += 1
xgb_rank['XGB_rank']    += 1
merged_imp = ridge_rank[['Feature','Ridge_rank','Coefficient']].merge(
    xgb_rank[['Feature','XGB_rank','Gain']], on='Feature', how='outer'
).sort_values('XGB_rank').fillna({'Ridge_rank': 99})
merged_imp['Ridge_rank'] = merged_imp['Ridge_rank'].astype(int)
print('\nFeature rank comparison (Ridge rank vs XGB rank):')
display(merged_imp.to_string(index=False))

Ridge — standardised coefficients (features scaled to unit variance):
  Large team_ortg/drtg_ctx coefficients → Ridge is partly predicting team quality


,Feature,Coefficient,AbsCoef
0,team_ortg_ctx,6.360,6.360
1,team_drtg_ctx,-5.780,5.780
2,Creation,1.276,1.276
3,Load,-1.223,1.223
4,Deflections_per100,0.477,0.477
5,RimPointsSaved,0.462,0.462
6,PTS_per100,0.328,0.328
7,cTOV_pct,-0.298,0.298
8,GP_pct,-0.230,0.230
9,PasserRating,0.228,0.228



XGBoost — feature gain (captures non-linear importance, more player-specific):


,Feature,Gain
0,zTS,57.005
1,team_ortg_ctx,56.450
2,Creation,54.547
3,PTS_per100,40.928
4,RimPointsSaved,30.278
5,Deflections_per100,26.411
6,Load,25.271
7,PasserRating,22.824
8,DREB_Uncontest_per100,18.654
9,RimAssists_per100,17.949



Feature rank comparison (Ridge rank vs XGB rank):


'              Feature  Ridge_rank  Coefficient  XGB_rank   Gain\n                  zTS          15        0.161         1 57.005\n        team_ortg_ctx           1        6.360         2 56.450\n             Creation           3        1.276         3 54.547\n           PTS_per100           7        0.328         4 40.928\n       RimPointsSaved           6        0.462         5 30.278\n   Deflections_per100           5        0.477         6 26.411\n                 Load           4       -1.223         7 25.271\n         PasserRating          10        0.228         8 22.824\nDREB_Uncontest_per100          11        0.224         9 18.654\n    RimAssists_per100          14        0.204        10 17.949\n             cTOV_pct           8       -0.298        11 17.863\n          fTOV_per100          19       -0.088        12 16.461\n          C6_Diff_pct          21        0.053        13 15.832\n             ThreePtP          13        0.212        14 15.572\n  OREB_Contest_per100   

## 5. 1-Year Inference (2024) + Ridge vs XGB Player Comparison

In [8]:
latest = rolled[(rolled['Season'] == 2024) & (rolled['Minutes'] >= 200)].copy()

# --- Total RAPM predictions ---
Xl = latest[avail].copy()
for col in Xl.columns:
    fv = gmed.get(col, 0.0)
    Xl[col] = Xl[col].fillna(0.0 if pd.isna(fv) else fv)

latest['Box_Ridge'] = lin_m.predict(lin_sc.transform(Xl.values))
latest['Box_XGB']   = xgb_m.predict(Xl.values)
latest['Box_Avg']   = (latest['Box_Ridge'] + latest['Box_XGB']) / 2

# --- Separate O / D predictions (for prior.csv export) ---
Xl_o = latest[[c for c in o_avail if c in latest.columns]].copy()
for col in Xl_o.columns:
    Xl_o[col] = Xl_o[col].fillna(gmed_o.get(col, 0.0))
latest['Box_ORAPM'] = (o_lin_m.predict(o_lin_sc.transform(Xl_o.values)) +
                       o_xgb_m.predict(Xl_o.values)) / 2

Xl_d = latest[[c for c in d_avail if c in latest.columns]].copy()
for col in Xl_d.columns:
    Xl_d[col] = Xl_d[col].fillna(gmed_d.get(col, 0.0))
latest['Box_DRAPM'] = (d_lin_m.predict(d_lin_sc.transform(Xl_d.values)) +
                       d_xgb_m.predict(Xl_d.values)) / 2

# --- Export prior.csv for rapm_with_prior.py ---
prior_out = HERE / 'data' / 'processed' / 'prior_export.csv'
prior_df  = write_prior_csv(latest, 'Box_ORAPM', 'Box_DRAPM', 2024, out_path=prior_out)
print('To run proper RAPM-with-prior:')
print(f'  cp {prior_out} {JE}/prior.csv && cd {JE} && python rapm_with_prior.py')
print()

# --- Approximate blend with α=6000 (best per retrodiction) ---
BEST_ALPHA = 6000
latest = compute_prior_blend(latest, pred_col='Box_Ridge', raw1y_col='RAPM_1y',
                              alpha=BEST_ALPHA, out_col='Final_Ridge')
latest = compute_prior_blend(latest, pred_col='Box_XGB', raw1y_col='RAPM_1y',
                              alpha=BEST_ALPHA, out_col='Final_XGB')
latest['Final_Avg'] = (latest['Final_Ridge'] + latest['Final_XGB']) / 2

# Add ranks for comparison
latest['Ridge_rank'] = latest['Box_Ridge'].rank(ascending=False).astype(int)
latest['XGB_rank']   = latest['Box_XGB'].rank(ascending=False).astype(int)
latest['Rank_diff']  = latest['Ridge_rank'] - latest['XGB_rank']  # positive = XGB rates higher

# --- Side-by-side: Ridge top 25 ---
print('=== Box Model — Ridge top 25 ===')
display(latest.nlargest(25, 'Box_Ridge')[
    ['Player','Minutes','Box_Ridge','Box_XGB','XGB_rank','Rank_diff','RAPM_3y','RAPM_1y']
].rename(columns={'XGB_rank':'XGB_rank_of_player', 'Rank_diff':'Ridge_minus_XGB_rank'}))

print('\n=== Box Model — XGBoost top 25 ===')
display(latest.nlargest(25, 'Box_XGB')[
    ['Player','Minutes','Box_XGB','Box_Ridge','Ridge_rank','Rank_diff','RAPM_3y','RAPM_1y']
].rename(columns={'Ridge_rank':'Ridge_rank_of_player', 'Rank_diff':'Ridge_minus_XGB_rank'}))

print(f'\n=== Final (Box+1yr prior α={BEST_ALPHA}) top 25 ===')
display(latest.nlargest(25, 'Final_Avg')[
    ['Player','Team','Minutes','Box_Ridge','Box_XGB','Box_ORAPM','Box_DRAPM','Final_Ridge','Final_XGB','Final_Avg','RAPM_3y','RAPM_1y']
])

  Wrote 436 player priors for season 2024 → /Users/eadebayo/Documents/Projects/Sports Analytics/New SPM/zts/data/processed/prior_export.csv
To run proper RAPM-with-prior:
  cp /Users/eadebayo/Documents/Projects/Sports Analytics/New SPM/zts/data/processed/prior_export.csv /Users/eadebayo/Documents/Projects/Sports Analytics/New SPM/JE_rapm/prior.csv && cd /Users/eadebayo/Documents/Projects/Sports Analytics/New SPM/JE_rapm && python rapm_with_prior.py

=== Box Model — Ridge top 25 ===


,Player,Minutes,Box_Ridge,Box_XGB,XGB_rank_of_player,Ridge_minus_XGB_rank,RAPM_3y,RAPM_1y
5316,Nikola Jokić,2737.000,7.255,7.649,1,0,8.821,6.187
5442,Luka Dončić,2624.000,5.240,3.338,23,-21,3.572,4.284
5369,Jayson Tatum,2645.000,5.053,5.790,4,-1,5.701,3.128
5223,Kevin Durant,2791.000,4.862,5.389,7,-3,5.217,1.582
5309,Joel Embiid,1309.000,4.721,6.601,2,3,7.214,4.738
5238,James Harden,2470.000,4.371,2.716,37,-31,2.556,2.231
5239,Stephen Curry,2421.000,4.139,5.336,8,-1,5.156,1.838
5219,LeBron James,2504.000,3.914,5.260,9,-1,5.073,4.842
5292,Giannis Antetokounmpo,2567.000,3.802,6.277,3,6,6.353,4.132
5260,Kawhi Leonard,2330.000,3.784,5.552,6,4,5.589,4.225



=== Box Model — XGBoost top 25 ===


,Player,Minutes,Box_XGB,Box_Ridge,Ridge_rank_of_player,Ridge_minus_XGB_rank,RAPM_3y,RAPM_1y
5316,Nikola Jokić,2737.000,7.649,7.255,1,0,8.821,6.187
5309,Joel Embiid,1309.000,6.601,4.721,5,3,7.214,4.738
5292,Giannis Antetokounmpo,2567.000,6.277,3.802,9,6,6.353,4.132
5369,Jayson Tatum,2645.000,5.790,5.053,3,-1,5.701,3.128
5250,Paul George,2502.000,5.635,3.163,25,20,6.301,5.710
5260,Kawhi Leonard,2330.000,5.552,3.784,10,4,5.589,4.225
5223,Kevin Durant,2791.000,5.389,4.862,4,-3,5.217,1.582
5239,Stephen Curry,2421.000,5.336,4.139,7,-1,5.156,1.838
5219,LeBron James,2504.000,5.260,3.914,8,-1,5.073,4.842
5242,Jrue Holiday,2263.000,5.004,3.632,11,1,5.875,1.750



=== Final (Box+1yr prior α=6000) top 25 ===


,Player,Team,Minutes,Box_Ridge,Box_XGB,Box_ORAPM,Box_DRAPM,Final_Ridge,Final_XGB,Final_Avg,RAPM_3y,RAPM_1y
5316,Nikola Jokić,DEN,2737.000,7.255,7.649,6.087,-0.887,6.739,6.943,6.841,8.821,6.187
5309,Joel Embiid,PHI,1309.000,4.721,6.601,4.108,-1.126,4.727,6.026,5.376,7.214,4.738
5250,Paul George,LAC,2502.000,3.163,5.635,2.887,-1.119,4.337,5.670,5.003,6.301,5.710
5219,LeBron James,LAL,2504.000,3.914,5.260,4.280,-0.759,4.342,5.067,4.705,5.073,4.842
5292,Giannis Antetokounmpo,MIL,2567.000,3.802,6.277,3.037,-1.891,3.956,5.275,4.616,6.353,4.132
5387,Derrick White,BOS,2381.000,3.607,3.941,1.904,-0.822,4.432,4.616,4.524,5.222,5.446
5260,Kawhi Leonard,LAC,2330.000,3.784,5.552,3.716,-0.520,3.980,4.964,4.472,5.589,4.225
5414,Shai Gilgeous-Alexander,OKC,2553.000,2.280,4.272,3.399,-0.538,3.928,4.993,4.460,5.182,5.818
5369,Jayson Tatum,BOS,2645.000,5.053,5.790,3.949,-0.233,4.139,4.526,4.333,5.701,3.128
5442,Luka Dončić,DAL,2624.000,5.240,3.338,4.820,0.186,4.788,3.785,4.287,3.572,4.284


## 6. Retrodiction Test — Box Model Only (Metric A)

Predict year-Y player RAPM from box features → apply to Y+1 minutes → team Pearson r² vs actual wins.

In [9]:
TEST_SEASONS = list(range(2017, 2024))   # predict 2017–2023, test on 2018–2024 wins

ret_lin = retrodiction_test(lin_m, rolled, tw, avail,
    scaler=lin_sc, replacement_value=repl_val, test_seasons=TEST_SEASONS,
    label='Ridge (box only)', global_medians=gmed)

ret_xgb = retrodiction_test(xgb_m, rolled, tw, avail,
    scaler=None, replacement_value=repl_val, test_seasons=TEST_SEASONS,
    label='XGB (box only)', global_medians=gmed)

display(pd.concat([ret_lin, ret_xgb]).pivot(index='Season_predicted', columns='Label', values='R2').round(4))

Label,Ridge (box only),XGB (box only)
Season_predicted,,
2017,0.564,0.605
2018,0.526,0.651
2019,0.610,0.704
2020,0.540,0.511
2021,0.557,0.690
2022,0.573,0.659
2023,0.458,0.555


## 7. Prior Blend Retrodiction — Box + 1yr RAPM (Metric B)

Box model prediction used as Bayesian prior → shrunk toward 1-year raw RAPM → team r² vs wins.

In [10]:
def score_all(model, scaler, medians, feat_cols):
    """Apply a trained model to all seasons in `rolled`."""
    r = rolled.copy()
    X2 = r[[c for c in feat_cols if c in r.columns]].copy()
    for col in X2.columns:
        X2[col] = X2[col].fillna(medians.get(col, 0.0))
    Xa = scaler.transform(X2.values) if scaler else X2.values
    r['box_pred'] = model.predict(Xa)
    return r

rolled_ridge = score_all(lin_m, lin_sc, gmed, avail)
rolled_xgb   = score_all(xgb_m, None,   gmed, avail)

# Also keep a reference for the team detail cell below
rolled_xgb_pr = compute_prior_blend(rolled_xgb, pred_col='box_pred',
                                     raw1y_col='RAPM_1y', alpha=4000, out_col='RAPM_blended')

# Test α = 2000, 4000, 6000
ALPHAS = [2000, 4000, 6000]
all_blend_results = []

for alpha in ALPHAS:
    for base_rolled, model_tag in [(rolled_ridge, 'Ridge'), (rolled_xgb, 'XGB')]:
        blended = compute_prior_blend(base_rolled, pred_col='box_pred',
                                      raw1y_col='RAPM_1y',
                                      alpha=alpha, out_col='RAPM_blended')
        label = f'{model_tag}+1yr α={alpha}'
        ret   = retrodiction_test(None, blended, tw, avail, score_col='RAPM_blended',
                                  replacement_value=repl_val, test_seasons=TEST_SEASONS,
                                  label=label)
        all_blend_results.append(ret)

blend_df = pd.concat(all_blend_results, ignore_index=True)

print('=== Blended retrodiction r² (α comparison) ===')
pivot_alpha = blend_df.pivot(index='Season_predicted', columns='Label', values='R2').round(4)
display(pivot_alpha)

print('\n=== Average r² by alpha ===')
avg_alpha = blend_df.groupby('Label')['R2'].mean().reset_index()
avg_alpha.columns = ['Model', 'Avg r²']
display(avg_alpha.sort_values('Avg r²', ascending=False).round(4))

=== Blended retrodiction r² (α comparison) ===


Label,Ridge+1yr α=2000,Ridge+1yr α=4000,Ridge+1yr α=6000,XGB+1yr α=2000,XGB+1yr α=4000,XGB+1yr α=6000
Season_predicted,,,,,,
2017,0.648,0.631,0.620,0.661,0.651,0.643
2018,0.602,0.593,0.585,0.654,0.665,0.668
2019,0.650,0.666,0.669,0.643,0.671,0.684
2020,0.497,0.525,0.537,0.475,0.496,0.505
2021,0.505,0.531,0.543,0.532,0.580,0.607
2022,0.648,0.646,0.639,0.680,0.694,0.696
2023,0.577,0.567,0.554,0.594,0.603,0.602



=== Average r² by alpha ===


,Model,Avg r²
5,XGB+1yr α=6000,0.629
4,XGB+1yr α=4000,0.623
3,XGB+1yr α=2000,0.606
1,Ridge+1yr α=4000,0.594
2,Ridge+1yr α=6000,0.592
0,Ridge+1yr α=2000,0.590


## 8. Combined Results + Plot

In [11]:
all_retro = pd.concat([ret_lin, ret_xgb, blend_df], ignore_index=True)

print('=== Retrodiction Pearson r² — All Models ===')
pivot = all_retro.pivot(index='Season_predicted', columns='Label', values='R2')
display(pivot.round(4))

print('\n=== Average r² Summary ===')
summary = all_retro.groupby('Label')['R2'].mean().sort_values(ascending=False).reset_index()
summary.columns = ['Model','Avg r²']
display(summary.round(4))

=== Retrodiction Pearson r² — All Models ===


Label,Ridge (box only),Ridge+1yr α=2000,Ridge+1yr α=4000,Ridge+1yr α=6000,XGB (box only),XGB+1yr α=2000,XGB+1yr α=4000,XGB+1yr α=6000
Season_predicted,,,,,,,,
2017,0.564,0.648,0.631,0.620,0.605,0.661,0.651,0.643
2018,0.526,0.602,0.593,0.585,0.651,0.654,0.665,0.668
2019,0.610,0.650,0.666,0.669,0.704,0.643,0.671,0.684
2020,0.540,0.497,0.525,0.537,0.511,0.475,0.496,0.505
2021,0.557,0.505,0.531,0.543,0.690,0.532,0.580,0.607
2022,0.573,0.648,0.646,0.639,0.659,0.680,0.694,0.696
2023,0.458,0.577,0.567,0.554,0.555,0.594,0.603,0.602



=== Average r² Summary ===


,Model,Avg r²
0,XGB+1yr α=6000,0.629
1,XGB (box only),0.625
2,XGB+1yr α=4000,0.623
3,XGB+1yr α=2000,0.606
4,Ridge+1yr α=4000,0.594
5,Ridge+1yr α=6000,0.592
6,Ridge+1yr α=2000,0.590
7,Ridge (box only),0.547


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: time series — box-only vs best blended per model
box_styles  = [('Ridge (box only)', 'steelblue',  'o', '-'),
               ('XGB (box only)',   'darkorange', 's', '-')]
alpha_colors = {2000: '#2ca02c', 4000: '#9467bd', 6000: '#e377c2'}
alpha_marks  = {2000: '^', 4000: 'D', 6000: 'v'}

ax = axes[0]
for lab, color, marker, ls in box_styles:
    sub = all_retro[all_retro['Label']==lab].sort_values('Season_predicted')
    if not sub.empty:
        ax.plot(sub['Season_predicted'], sub['R2'],
                marker=marker, color=color, linestyle=ls, linewidth=2, markersize=6,
                label=f'{lab} ({sub.R2.mean():.3f})')

# Best blend per alpha (XGB, as it tends to win)
for alpha in ALPHAS:
    lab = f'XGB+1yr α={alpha}'
    sub = blend_df[blend_df['Label']==lab].sort_values('Season_predicted')
    if not sub.empty:
        ax.plot(sub['Season_predicted'], sub['R2'],
                marker=alpha_marks[alpha], color=alpha_colors[alpha],
                linestyle='--', linewidth=1.8, markersize=6,
                label=f'{lab} ({sub.R2.mean():.3f})')

ax.axhline(0.40, color='gray', linestyle=':', alpha=0.5, label='~BPM baseline')
ax.set_xlabel('Season predicted'); ax.set_ylabel('Pearson r²')
ax.set_title('Retrodiction: Pearson r² by Season\n(actual wins, actual next-year minutes)')
ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(0, 0.85)

# Right: average r² bar chart for all models
avg = all_retro.groupby('Label')['R2'].mean().sort_values(ascending=False)
bar_colors = []
hatches    = []
for lab in avg.index:
    if 'Ridge' in lab and 'α' not in lab:  bar_colors.append('steelblue');  hatches.append('')
    elif 'XGB' in lab and 'α' not in lab:  bar_colors.append('darkorange'); hatches.append('')
    elif 'α=2000' in lab:                  bar_colors.append('#2ca02c');    hatches.append('///')
    elif 'α=4000' in lab:                  bar_colors.append('#9467bd');    hatches.append('///')
    else:                                  bar_colors.append('#e377c2');    hatches.append('///')

bars = axes[1].bar(range(len(avg)), avg.values, color=bar_colors, edgecolor='k', linewidth=0.5)
for bar, h in zip(bars, hatches):
    bar.set_hatch(h)
axes[1].set_xticks(range(len(avg)))
axes[1].set_xticklabels(avg.index, rotation=25, ha='right', fontsize=7.5)
axes[1].set_ylabel('Average Pearson r²')
axes[1].set_title('Average Retrodiction r² — All Models & Alphas')
axes[1].axhline(0.40, color='gray', linestyle=':', alpha=0.5)
axes[1].grid(alpha=0.2, axis='y')
for i, v in enumerate(avg.values):
    axes[1].text(i, v + 0.003, f'{v:.3f}', ha='center', fontsize=7.5)

plt.tight_layout()
out_png = HERE / 'data' / 'processed' / 'retrodiction_r2.png'
plt.savefig(out_png, dpi=130)
plt.show()
print(f'Saved {out_png}')

Saved /Users/eadebayo/Documents/Projects/Sports Analytics/New SPM/zts/data/processed/retrodiction_r2.png


## 9. 2024 Leaderboard — All Metrics

In [13]:
show = latest.nlargest(30,'Final_Avg')[[
    'Player','Team','Minutes',
    'RAPM_3y',    # actual 3-yr RAPM (target)
    'RAPM_1y',    # actual 1-yr raw RAPM
    'Box_Ridge',  # box model prediction (Ridge)
    'Box_XGB',    # box model prediction (XGBoost)
    'Final_Avg',  # ensemble blended final RAPM
    # Key features
    'Creation','zTS','RimPointsSaved','fTOV_per100',
]]
print('2024 Top 30 — Final RAPM (box model + 1yr prior blend):')
display(show)

2024 Top 30 — Final RAPM (box model + 1yr prior blend):


,Player,Team,Minutes,RAPM_3y,RAPM_1y,Box_Ridge,Box_XGB,Final_Avg,Creation,zTS,RimPointsSaved,fTOV_per100
5316,Nikola Jokić,DEN,2737.000,8.821,6.187,7.255,7.649,6.841,15.386,8.397,0.044,5.384
5309,Joel Embiid,PHI,1309.000,7.214,4.738,4.721,6.601,5.376,11.866,6.470,1.463,3.457
5250,Paul George,LAC,2502.000,6.301,5.710,3.163,5.635,5.003,8.630,3.480,0.529,6.157
5219,LeBron James,LAL,2504.000,5.073,4.842,3.914,5.260,4.705,12.385,3.883,0.560,3.634
5292,Giannis Antetokounmpo,MIL,2567.000,6.353,4.132,3.802,6.277,4.616,10.811,4.457,1.692,3.402
5387,Derrick White,BOS,2381.000,5.222,5.446,3.607,3.941,4.524,5.635,2.760,0.984,3.688
5260,Kawhi Leonard,LAC,2330.000,5.589,4.225,3.784,5.552,4.472,8.279,6.653,0.165,5.342
5414,Shai Gilgeous-Alexander,OKC,2553.000,5.182,5.818,2.280,4.272,4.460,10.119,5.477,0.188,5.451
5369,Jayson Tatum,BOS,2645.000,5.701,3.128,5.053,5.790,4.333,8.769,4.473,0.262,3.342
5442,Luka Dončić,DAL,2624.000,3.572,4.284,5.240,3.338,4.287,17.123,5.710,-0.278,4.429


## 10. Team Retrodiction Detail (2023 → 2024)

In [14]:
PRED_YEAR = 2023; TEST_YEAR = 2024

# Score 2023 with XGB+Prior
df_pred = rolled_xgb_pr[rolled_xgb_pr['Season']==PRED_YEAR].copy()
score_map = dict(zip(df_pred['PLAYER_ID'], df_pred['RAPM_blended']))

df_test = rolled_xgb_pr[(rolled_xgb_pr['Season']==TEST_YEAR)].dropna(subset=['Team']).copy()
df_test['raw_score'] = df_test['PLAYER_ID'].map(score_map)
df_test['adj_score'] = np.where(df_test['Minutes']<200, repl_val,
                                 df_test['raw_score'].fillna(rookie_val))
df_test['contribution'] = df_test['adj_score'] * df_test['Minutes']

team_scores = df_test.groupby('Team')['contribution'].sum().rename('team_score').reset_index()
tc = team_scores.merge(tw[tw.Season==TEST_YEAR][['Team','team_wins']], on='Team')
tc['Score_Rank'] = tc['team_score'].rank(ascending=False).astype(int)
tc['Wins_Rank']  = tc['team_wins'].rank(ascending=False).astype(int)

r_val = tc[['team_wins','team_score']].corr().iloc[0,1]
print(f'{PRED_YEAR}→{TEST_YEAR} XGB+Prior:  Pearson r = {r_val:.4f},  r² = {r_val**2:.4f}\n')
display(tc.sort_values('team_wins', ascending=False)[['Team','team_wins','Wins_Rank','team_score','Score_Rank']])

2023→2024 XGB+Prior:  Pearson r = 0.7764,  r² = 0.6028



,Team,team_wins,Wins_Rank,team_score,Score_Rank
1,BOS,64,1,48042.647,1
18,OKC,57,2,8694.257,20
5,DEN,57,2,33526.314,2
15,MIN,56,4,26021.425,7
10,LAC,51,5,27799.928,4
17,NYK,50,6,24194.772,8
4,DAL,50,6,20359.530,11
21,PHO,49,9,23311.327,9
16,NOP,49,9,21343.442,10
14,MIL,49,9,26994.748,5


## 11. Save Outputs

In [15]:
out_dir = HERE / 'data' / 'processed'
out_dir.mkdir(parents=True, exist_ok=True)

# Retrodiction summary (all models + alphas)
all_retro.to_csv(out_dir / 'retrodiction_results.csv', index=False)

# 2024 player scores
save_cols = ['Player','PLAYER_ID','Season','Team','Minutes','RAPM_3y','RAPM_1y',
             'Box_Ridge','Box_XGB','Box_ORAPM','Box_DRAPM',
             'Final_Ridge','Final_XGB','Final_Avg'] + avail
latest[[c for c in save_cols if c in latest.columns]].sort_values('Final_Avg',ascending=False).to_csv(
    out_dir / 'spm_2024.csv', index=False)

# Rolled features + prior CSV (already written by write_prior_csv above)
rolled.to_csv(out_dir / 'rolled_features.csv', index=False)

print('=== Files saved ===')
for f in ['retrodiction_results.csv','spm_2024.csv','rolled_features.csv',
          'retrodiction_r2.png','prior_export.csv']:
    p = out_dir / f
    if p.exists():
        print(f'  {f}  ({p.stat().st_size/1024:.1f} KB)')

print(f'\n=== Final Summary ===')
print(f'Training player-seasons:  {X.shape[0]:,}')
print(f'Features (total model):   {X.shape[1]}')
print(f'Replacement RAPM:         {repl_val:.3f}')
print(f'Alphas tested:            {ALPHAS}')
print()
for _, row in summary.iterrows():
    print(f"  {row['Model']:<28} avg r² = {row['Avg r²']:.4f}")

print('\n=== To run proper RAPM-with-Prior (requires MySQL) ===')
print(f'  cp {out_dir}/prior_export.csv {JE}/prior.csv')
print(f'  cd {JE} && python rapm_with_prior.py')

=== Files saved ===
  retrodiction_results.csv  (2.3 KB)
  spm_2024.csv  (217.3 KB)
  rolled_features.csv  (3022.9 KB)
  retrodiction_r2.png  (181.9 KB)
  prior_export.csv  (13.3 KB)

=== Final Summary ===
Training player-seasons:  3,823
Features (total model):   22
Replacement RAPM:         -0.283
Alphas tested:            [2000, 4000, 6000]

  XGB+1yr α=6000               avg r² = 0.6293
  XGB (box only)               avg r² = 0.6249
  XGB+1yr α=4000               avg r² = 0.6229
  XGB+1yr α=2000               avg r² = 0.6056
  Ridge+1yr α=4000             avg r² = 0.5943
  Ridge+1yr α=6000             avg r² = 0.5924
  Ridge+1yr α=2000             avg r² = 0.5896
  Ridge (box only)             avg r² = 0.5470

=== To run proper RAPM-with-Prior (requires MySQL) ===
  cp /Users/eadebayo/Documents/Projects/Sports Analytics/New SPM/zts/data/processed/prior_export.csv /Users/eadebayo/Documents/Projects/Sports Analytics/New SPM/JE_rapm/prior.csv
  cd /Users/eadebayo/Documents/Projects/Spo